# HawkEye RAG — Day 2

## Real chunking + a persistent vector store

Today: load the full 8-category knowledge base with frontmatter stripped, chunk it, embed with Gemini (`gemini-embedding-001`), store in Chroma, then visualize with t-SNE.

Run cells top to bottom, in order. Each cell says what it depends on.

In [1]:
# --- Setup: imports, env, config ---
# Run this first, always. If you ever restart the kernel, re-run this cell before anything else.

from pathlib import Path
import os

import frontmatter
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

from tenacity import retry, wait_exponential, stop_after_attempt
from tqdm import tqdm

import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import plotly.io as pio

# Fix VS Code / plotly renderer detection quirk up front, before it can bite us later.
pio.renderers.default = "vscode"

load_dotenv(override=True)

KNOWLEDGE_BASE_PATH = Path("knowledge-base")
DB_NAME = "vector_db"
EMBEDDING_MODEL = "gemini-embedding-001"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
BATCH_SIZE = 100

print("Setup complete.")

Setup complete.


### Part A — Load and chunk

In [2]:
# --- Load every article, stripping YAML frontmatter ---
# Depends on: setup cell above.

def fetch_documents():
    """
    Walk every category folder in the knowledge base, strip YAML frontmatter
    from each .md file, and return one dict per article.
    """
    documents = []

    for category_folder in KNOWLEDGE_BASE_PATH.iterdir():
        if not category_folder.is_dir():
            continue
        doc_type = category_folder.name

        for file_path in category_folder.rglob("*.md"):
            post = frontmatter.load(file_path)
            documents.append({
                "type": doc_type,
                "source": file_path.as_posix(),
                "text": post.content,             # clean body, frontmatter stripped
                "metadata": dict(post.metadata),   # title, tags, url, etc.
            })

    print(f"Loaded {len(documents)} documents")
    return documents

documents = fetch_documents()

Loaded 854 documents


In [3]:
# --- Sanity check: confirm frontmatter really was stripped out ---
# Depends on: documents (previous cell). Optional to run, but cheap and worth eyeballing.

documents[0]

{'type': 'Accounts-Access-Security',
 'source': 'knowledge-base/Accounts-Access-Security/2017-05-25-Fraudulent-message-with-subject-Vital-Information-31078.md',
 'text': 'This message is fraudulent email that was received by some faculty and staff on May 25th. \xa0The goal is to get people to open the attachment, click the link there, and go to a site and provide their username & password. \xa0It is a common, though not extremely sophisticated, type of scam.\n\nRed Flags: *(These are things that not only appear in this message, but are commonly used in fraudulent messages. \xa0You should be on the lookout for these kinds of tricks in other emails as well)*\n\n* Generic message text: \xa0The message is *extremely* generic. \xa0There\'s no actual description or indication of what the message is about.\n* Generic attachment name:\xa0There\'s no description of what the attachment is in the message and the file name is vague\n* False sense of importance/urgency: Just "Vital" information.\n\

In [4]:
# --- Wrap into LangChain Documents, then chunk ---
# Depends on: documents.

langchain_docs = [
    Document(
        page_content=doc["text"],
        metadata={
            "doc_type": doc["type"],
            "source": doc["source"],
            "title": doc["metadata"].get("title", ""),
            "url": doc["metadata"].get("url", ""),
        }
    )
    for doc in documents
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = text_splitter.split_documents(langchain_docs)
print(f"Divided into {len(chunks)} chunks")

Divided into 7676 chunks


### Part B — Embed and store in Chroma

This step makes ~80 API calls to Gemini and can take a couple of minutes. It's written to be **safe to re-run**: if it gets interrupted (rate limit, network blip, you closing the laptop), just re-run the same cell — it checks how much is already in the store and picks up where it left off instead of re-embedding from scratch.

In [5]:
# --- Embed and store, resumable ---
# Depends on: chunks. Safe to re-run if interrupted.

embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

wait = wait_exponential(multiplier=1, min=10, max=120)

@retry(wait=wait, stop=stop_after_attempt(6))
def add_batch(batch):
    vectorstore.add_documents(batch)

existing_count = vectorstore._collection.count()
start = (existing_count // BATCH_SIZE) * BATCH_SIZE  # resume at the last complete batch

if existing_count >= len(chunks):
    print(f"Store already has {existing_count} documents — nothing to do.")
else:
    print(f"Resuming from chunk {start} (store currently has {existing_count} documents)")
    for i in tqdm(range(start, len(chunks), BATCH_SIZE)):
        batch = chunks[i:i + BATCH_SIZE]
        add_batch(batch)

print(f"Vectorstore now has {vectorstore._collection.count()} documents")

Store already has 7676 documents — nothing to do.
Vectorstore now has 7676 documents


In [6]:
# --- Inspect the store: count + dimensionality ---
# Depends on: vectorstore.

collection = vectorstore._collection
count = collection.count()

sample = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample)
print(f"{count:,} vectors with {dimensions:,} dimensions")

7,676 vectors with 3,072 dimensions


### Part C — Visualize!

In [7]:
# --- Pull every embedding back out of Chroma, prep colors ---
# Depends on: collection.

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
doc_texts = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]

unique_types = sorted(set(doc_types))
palette = ['blue', 'green', 'red', 'orange', 'purple', 'brown', 'teal', 'magenta']
color_map = dict(zip(unique_types, palette))
colors = [color_map[t] for t in doc_types]

print(f"Categories found ({len(unique_types)}): {unique_types}")

Categories found (8): ['Accounts-Access-Security', 'Digital-Accessibility', 'Getting-Started-Guides', 'Hardware', 'Internal-Documentation', 'Networking-WiFi', 'Policies', 'Software-and-Apps']


In [8]:
# --- 2D t-SNE plot ---
# Depends on: vectors, doc_types, colors. Takes a minute or two on ~7,700 points.

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, doc_texts)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vector Store Visualization — HawkEye Knowledge Base',
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [9]:
# --- 3D t-SNE plot (optional, run after you've looked at the 2D one) ---
# Depends on: vectors, doc_types, colors. Takes a minute or two.

tsne_3d = TSNE(n_components=3, random_state=42)
reduced_vectors_3d = tsne_3d.fit_transform(vectors)

fig_3d = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors_3d[:, 0],
    y=reduced_vectors_3d[:, 1],
    z=reduced_vectors_3d[:, 2],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, doc_texts)],
    hoverinfo='text'
)])

fig_3d.update_layout(
    title='3D Chroma Vector Store Visualization — HawkEye Knowledge Base',
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig_3d.show()